# பாடம் 13 - முகவர் நினைவகம்


## அமைக்கவும்

இந்த நோட்புக் **Microsoft Agent Framework** (MAF) பயன்படுத்தி **நிலையான நினைவாற்றல்** கொண்ட பயண முகவர் உருவாக்குவதையை விளக்குகிறது.

முகவர் நினைவின் வெவ்வேறு வகைகள் — வேலை நினைவு, குறுகிய கால நினைவு, மற்றும் நீண்ட கால நினைவு — உரையாடல்களில் முகவர் தகவலை எவ்வாறு வைத்துக்கொள்ளும் மற்றும் பயன்படுத்துவதில் எப்படி தாக்கம் செலுத்துகின்றன என்பது பற்றி நீங்கள் கற்றுக்கொள்ளப்போகின்றீர்கள்.

**முந்தய தேவைகள்:**
- ஒருசிறந்த Microsoft Foundry திட்டம் மற்றும் அதில் பணியாற்றும் அரட்டை மாதிரி (உதா., `gpt-4.1-mini`).
- Azure CLI-ல் உள்நுழைந்திருத்தல் — உங்கள் டெர்மினலில் `az login` இயக்கவும்.
- `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Microsoft Foundry திட்ட புள்ளி முகவரி.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் பணிபுரியும் மாதிரியின் பெயர்.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## முகவர் நினைவக வகைகள்

செயற்கை நுண்ணறிவு முகவர்கள் வெவ்வேறு வகையான நினைவகங்களை பயன்படுத்த முடியும், ஒவ்வொன்றும் தனித்துவமான நோக்கத்திற்காக:

### வேலை நினைவகம்
உரையாடல் தொடரே — ஒரே அமர்வில் பரிமாறப்படும் செய்திகள். முகவர் ஒரே தொடரில் முந்தைய செய்திகளுக்கு திரும்பிக் காணலாம் மற்றும் ஒற்றுமையை பராமரிக்க முடியும். MAF இல் இது **`agent.create_session()`** மூலம் உருவாக்கப்படுகிறது, இது ஒரு `AgentSession` ஐ திருப்புகிறது.

### குறுகிய கால நினைவகம்
ஒரு பணியின் அல்லது அமர்வின் காலத்திற்கான தகவல்கள், ஆனால் நிரந்தரமாக சேமிக்கப்படாதவை. உதாரணமாக, முகவர் ஒரு பலமுறை திட்டமிடும் உரையாடல் போது உண்மைகள் சேகரித்து, அவற்றைப் பயன்படுத்தி இறுதித் திட்டத்தை வெளிப்படுத்தலாம்.

### நீண்டகால நினைவகம்
**அமர்வுகளுக்கு மத்தியில்** நீடிக்கும் விருப்பங்கள் மற்றும் உண்மைகள். திரும்ப வந்த பயனர் தங்களது உணவு பற்றிய வரம்புகள் அல்லது பயண முறையை மீண்டும் சொல்ல வேண்டாம். நீண்டகால நினைவகம் பொதுவாக வெளியே உள்ள சேமிப்பகத்தால் ஆதரிக்கப்படுகிறது — தரவுத்தளம், கோப்பு அல்லது வெக்டர் குறியீடு — மற்றும் கருவிகளின் மூலம் முகவருக்கு வெளிப்படுத்தப்படுகிறது.


## அமர்வு மீதான பணிபுரியும் நினைவகம்ம்

நினைவகத்தின் எளிய வடிவம் உரையாடல் அமர்வு ஆகும். நீங்கள் ஒரே அமர்வை (`agent.create_session()` மூலம் உருவாக்கப்பட்டது) தொடர்ச்சியான `agent.run()` அழைப்புகளுக்கு வழங்கும்போது, அந்த முகவர் அந்த உரையாடலின் முழு வரலாற்றையும் பார்க்க முடியும் மற்றும் முந்தைய விவரங்களை நினைவில் வைக்க முடியும்.

ஒரு பயணம் முகவரியை உருவாக்கி பணிபுரியும் நினைவகத்தை காட்டுவோம்.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

முகவர் சரியானபடி பட்ஜெட்டை நினைவில் வைத்திருந்தார் ஏனெனில் இரு செய்திகள் ஒரே அமர்வை பகிர்ந்துள்ளன. இது **செயல்பாட்டு நினைவு** — இது அமர்வு காலத்திற்கு மட்டுமே இருக்கும்.

### புதிய திரெடுடன் என்ன நடக்கும்?

நாம் **புதிய** அமர்வு உருவாக்கினால், முகவருக்கு முந்தைய உரையாடல் தொடர்பான நினைவு இல்லை:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## நீண்டகால நினைவுக் கொள்

பயனர் விருப்பங்களை **அறிகுறிகளை கடந்தே** நினைவில் வைக்க, உரையாடல் நூல்மூலமான குருதியைத் தாண்டி வாழும் ஒரு நிலையான சேமிப்பகம் தேவை. முகவர் இந்த சேமிப்பகத்தை அணுக **கருவிகள்** மூலமாக செயல் படுத்தும் — தகவலை சேமிப்பதற்கும் மீட்டெடுப்பதற்கும் அழைக்கக்கூடிய செயல்பாடுகள்.

கீழே நாங்கள் ஒரு எளிய நினைவக வீர்ப்பு சேமிப்பகத்தை செயல்படுத்துகிறோம் (உற்பத்தியில் இதை ஒரு தரவுத்தளத்தால் அல்லது வெக்டர் குறியீட்டால் ஆதரிக்க வேண்டும்) மற்றும் முகவரால் பயன்படுத்தக்கூடிய கருவிகளாக இதைக் காட்டு.

### கட்டமைப்பு
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### நிலை 1 — முதலாவது பயனர் ஆண்டுதோறும் பயணம் பதிவு செய்கிறார்

சாரா முதன்முறையாக வருகிறார். முகவர் 그녀 உட்புகுத்தல்கள் டூல்களின் மூலம் அவற்றை சேமிக்க வேண்டும் மற்றும் ஹோட்டல்களை பரிந்துரைக்க அவற்றைப் பயன்படுத்த வேண்டும்.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### காட்சி 2 — சாரா வாரங்களுக்கு பிறகு திரும்புகிறார்

சாரா ஒரு **புதிய த்ரெட்** (புதிய அமர்வை சிமுலேட் செய்கிறது) துவங்குகிறார். பணி நினைவகம் காலியாக உள்ளது, ஆனால் நீண்டகால விருப்பக் கடை இன்னும் அவரது தகவலை வைத்திருக்கிறது. முகவர் அதை மீட்டெடுத்து பரிந்துரைகளை தனிப்பயனாக்க பயன்படுத்த வேண்டும்.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் மூன்று வகை முகவர் நினைவகங்கள் மற்றும் அவற்றை Microsoft Agent Framework கொண்டு எப்படி செயல்படுத்துவது என்பதை கற்றுக் கொண்டீர்கள்:

| நினைவக வகை | MAF இயந்திரம் | வாழ்நாள் |
|---|---|---|
| **வேலைசெய்** | `agent.create_session()` | ஒரே உரையாடல் |
| **குறுகிய கால** | ஒரு திரையில் சேர்க்கப்பட்ட சூழல் | ஒரே பணிப் பொறுப்பு / அமர்வு |
| **நீண்ட கால** | `@tool` செயல்பாடுகளின் மூலம் அணுகப்படும் வெளி சேமிப்பு | அமர்வுகளுக்கு இடையே |

### முக்கிய கூற்றுகள்
1. **`agent.create_session()`** வேலை நினைவகத்தை வழங்குகிறது — முகவர் அமர்வின் முழு உரையாடல் வரலாற்றைப் பார்க்க முடியும்.
2. **புதிய அமர்வுகள் சூழலை இழக்கின்றன** — நீண்டகால நினைவகம் இல்லாமல், முகவர் கடந்த உரையாடல்களை நினைவில் வைத்துக்கொள்ள முடியாது.
3. **`@tool` செயல்பாடுகள்** அந்த இடைவெளியை நிரப்புகின்றன — அவை முகவருக்கு நிலையான சேமிப்பில் தகவல்களை சேமிக்கவும் பெறவும் அனுமதிக்கின்றன.
4. **தனிப்பயனாக்கல் காலத்துக்கும் மேம்படுகிறது** — மேலும் விருப்பங்கள் சேமிக்கப்பட்டால், முகவரின் பரிந்துரைகள் சிறந்ததாக இருக்கும்.

### நிஜ உலகப் பயன்பாடுகள்
- **வாடிக்கையாளர் சேவை**: வாடிக்கையாளர் வரலாறு மற்றும் விருப்பங்களை நினைவில் வைக்கவும்
- **தனிப்பட்ட உதவியாளர்கள்**: நாட்கள் அல்லது வாரங்களில் சூழலை பராமரிக்கவும்
- **ஆரோக்கிய சேவை**: நோயாளி தகவல்கள் மற்றும் விருப்பங்களை கண்காணிக்கவும்
- **மின்னவணிகம்**: வரலாறின் அடிப்படையில் தனிப்பயன் ஷாப்பிங்

### அடுத்த படிகள்
- நினைவக dict ஐ தரவுத்தளத்தோ அல்லது வெக்டர் சேமிப்பிடம் (உதாரணமாக Azure AI Search) மாற்றவும்
- காலத்துக்கு சொல்லப்பட்ட தகவலுக்கு நினைவக காலாவதியைச் சேர்க்கவும்
- பகிர்ந்த நினைவகத்துடன் பல முகவர் அமைப்புகளை கட்டமைக்கவும்
- அறிவு-கிராப் ஆதாரமான நினைவகத்துக்கு Cognee நோட்புக் ஆராயவும்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
